# Recovery Contribution Analysis

Checks whether the Recovery (tie-breaker) LLM call is actually earning its cost,
using your existing `pipeline_run.jsonl` — no re-run, no GPU needed.

Just set the path below to wherever your results file is on your laptop, then run all cells.

In [7]:
import json
from collections import Counter

# ---- EDIT THIS to point at your results file ----
PATH = "Data/llm_output/pipeline_run.jsonl"
# ---------------------------------------------------

VERDICT_TO_LABEL = {
    "correct": "optimal",
    "suboptimal": "valid_alternative",
    "incorrect": "incorrect",
}

def load_traces(path):
    traces = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                traces.append(json.loads(line))
    return traces

def norm(v):
    return (v or "").strip().lower()

def is_correct(verdict, gt):
    return VERDICT_TO_LABEL.get(norm(verdict)) == gt

traces = load_traces(PATH)
print(f"Loaded {len(traces)} traces from {PATH}")

Loaded 510 traces from Data/llm_output/pipeline_run.jsonl


## 1. How often did Recovery even get triggered?

In [ ]:
disagreements = [
    t for t in traces
    if t.get("recovery_flag") and t.get("ground_truth_label")
    and t.get("tutor_verdict") and t.get("verifier_verdict")
    and t.get("final_verdict")
]

n = len(disagreements)
print(f"Total traces:        {len(traces)}")
print(f"Disagreement cases:  {n}  ({100 * n / len(traces):.1f}% of all traces)")

## 2. The core question: does Recovery beat the cheapest free heuristic?

On the disagreement cases only, compare: always trust Tutor / always trust Verifier / what Recovery actually decided.

In [ ]:
tutor_right = sum(is_correct(t["tutor_verdict"], t["ground_truth_label"]) for t in disagreements)
verifier_right = sum(is_correct(t["verifier_verdict"], t["ground_truth_label"]) for t in disagreements)
recovery_right = sum(is_correct(t["final_verdict"], t["ground_truth_label"]) for t in disagreements)

print(f"On disagreement cases only (n={n}):")
print(f"  Always trust Tutor:     {tutor_right}/{n}  ({100*tutor_right/n:.1f}%)")
print(f"  Always trust Verifier:  {verifier_right}/{n}  ({100*verifier_right/n:.1f}%)")
print(f"  Recovery's actual call: {recovery_right}/{n}  ({100*recovery_right/n:.1f}%)")

best_cheap = max(tutor_right, verifier_right)
cheap_label = "Tutor" if tutor_right >= verifier_right else "Verifier"
delta = recovery_right - best_cheap

print(f"\nBest free baseline (always trust {cheap_label}): {best_cheap}/{n} ({100*best_cheap/n:.1f}%)")
if delta > 0:
    print(f"--> Recovery beats the best free heuristic by +{delta} cases (+{100*delta/n:.1f} pts). The extra call is earning its keep.")
elif delta == 0:
    print(f"--> Recovery ties the best free heuristic. Consider a cheaper rule instead of a 3rd LLM call.")
else:
    print(f"--> Recovery does WORSE than just trusting {cheap_label} by {-delta} cases ({100*delta/n:.1f} pts).")

## 3. What pattern does Recovery actually follow?

In [ ]:
sided_with_tutor = sum(1 for t in disagreements if norm(t["final_verdict"]) == norm(t["tutor_verdict"]))
sided_with_verifier = sum(1 for t in disagreements if norm(t["final_verdict"]) == norm(t["verifier_verdict"])
                           and norm(t["final_verdict"]) != norm(t["tutor_verdict"]))
sided_with_neither = n - sided_with_tutor - sided_with_verifier

print(f"Sided with Tutor:          {sided_with_tutor}/{n}  ({100*sided_with_tutor/n:.1f}%)")
print(f"Sided with Verifier:       {sided_with_verifier}/{n}  ({100*sided_with_verifier/n:.1f}%)")
print(f"Neither (new 3rd verdict): {sided_with_neither}/{n}  ({100*sided_with_neither/n:.1f}%)")

## 4. When Recovery agrees with a side, how often is that side actually right?

In [ ]:
def slice_accuracy(subset):
    if not subset:
        return None
    right = sum(is_correct(t["final_verdict"], t["ground_truth_label"]) for t in subset)
    return right, len(subset)

sided_tutor_cases = [t for t in disagreements if norm(t["final_verdict"]) == norm(t["tutor_verdict"])]
sided_verifier_cases = [t for t in disagreements if norm(t["final_verdict"]) == norm(t["verifier_verdict"])
                         and norm(t["final_verdict"]) != norm(t["tutor_verdict"])]

if sided_tutor_cases:
    r, tot = slice_accuracy(sided_tutor_cases)
    print(f"When Recovery sided with Tutor:    correct {r}/{tot} ({100*r/tot:.1f}%)")
if sided_verifier_cases:
    r, tot = slice_accuracy(sided_verifier_cases)
    print(f"When Recovery sided with Verifier: correct {r}/{tot} ({100*r/tot:.1f}%)")

## 5. Ground-truth label distribution among disagreement cases

In [ ]:
gt_dist = Counter(t["ground_truth_label"] for t in disagreements)
for label, count in gt_dist.most_common():
    print(f"  {label}: {count} ({100*count/n:.1f}%)")

## 6. Bottom line: net effect of Recovery on overall pipeline accuracy

Compares the real pipeline (with Recovery) against a simulated pipeline where disagreements just fall back to the cheapest free heuristic instead of a 3rd LLM call.

In [ ]:
agreements = [t for t in traces if not t.get("recovery_flag") and t.get("ground_truth_label") and t.get("final_verdict")]
agree_right = sum(is_correct(t["final_verdict"], t["ground_truth_label"]) for t in agreements)

total_all = len(agreements) + n
actual_total_right = agree_right + recovery_right
cheap_total_right = agree_right + best_cheap

print(f"Agreement cases (no Recovery needed): {agree_right}/{len(agreements)} correct")
print(f"Actual pipeline (with Recovery):        {actual_total_right}/{total_all}  ({100*actual_total_right/total_all:.2f}%)")
print(f"Simulated no-Recovery (always trust {cheap_label} on disagreement): {cheap_total_right}/{total_all}  ({100*cheap_total_right/total_all:.2f}%)")
print(f"\nNet effect of Recovery on overall accuracy: {100*actual_total_right/total_all - 100*cheap_total_right/total_all:+.2f} pts")